# Sign Language Recognition: CNN vs Vision Transformer (ViT)

This notebook builds and compares two models for classifying American Sign Language (ASL) hand gestures from images:
1. A baseline **CNN**
2. A fine-tuned **Vision Transformer (ViT)**

**Dataset:** [Sign Language MNIST](https://www.kaggle.com/datasets/datamunge/sign-language-mnist) (24 classes, A-Y excluding J and Z, 28x28 grayscale images)

Run this in **Google Colab** with a free GPU: Runtime → Change runtime type → GPU

## 1. Setup

In [ ]:
!pip install -q timm kagglehub

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 2. Download the dataset

This uses `kagglehub` to pull the dataset directly — no manual upload needed.

In [ ]:
import kagglehub

path = kagglehub.dataset_download('datamunge/sign-language-mnist')
print('Dataset downloaded to:', path)

train_df = pd.read_csv(f'{path}/sign_mnist_train.csv')
test_df = pd.read_csv(f'{path}/sign_mnist_test.csv')

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
train_df.head()

## 3. Prepare the data

In [ ]:
# Labels: 0-25 mapping to A-Z, but J(9) and Z(25) are excluded in this dataset
LABEL_MAP = {i: chr(65 + i) for i in range(26)}

def prepare_arrays(df):
    labels = df['label'].values
    images = df.drop('label', axis=1).values.reshape(-1, 28, 28).astype(np.uint8)
    return images, labels

train_images, train_labels = prepare_arrays(train_df)
test_images, test_labels = prepare_arrays(test_df)

num_classes = len(np.unique(train_labels))
print('Number of classes:', num_classes)

# Visualize a few samples
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(train_images[i], cmap='gray')
    ax.set_title(LABEL_MAP[train_labels[i]])
    ax.axis('off')
plt.show()

In [ ]:
class SignLanguageDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        img = np.stack([img] * 3, axis=-1)  # grayscale -> 3 channels for pretrained models
        label = int(self.labels[idx])
        if self.transform:
            img = self.transform(img)
        return img, label

# Two transform pipelines: small for CNN, larger (224x224) for ViT
cnn_transform = transforms.Compose([
    transforms.ToTensor(),
])

vit_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

BATCH_SIZE = 64

## 4. Baseline CNN Model

In [ ]:
train_ds_cnn = SignLanguageDataset(train_images, train_labels, cnn_transform)
test_ds_cnn = SignLanguageDataset(test_images, test_labels, cnn_transform)

train_loader_cnn = DataLoader(train_ds_cnn, batch_size=BATCH_SIZE, shuffle=True)
test_loader_cnn = DataLoader(test_ds_cnn, batch_size=BATCH_SIZE, shuffle=False)

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = SimpleCNN(num_classes=26).to(device)
print(cnn_model)

In [ ]:
def train_model(model, train_loader, test_loader, epochs=8, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'test_acc': []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Evaluate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                outputs = model(images)
                preds = outputs.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.numpy())

        acc = accuracy_score(all_labels, all_preds)
        avg_loss = running_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['test_acc'].append(acc)
        print(f'Epoch {epoch+1}/{epochs} | Loss: {avg_loss:.4f} | Test Accuracy: {acc:.4f}')

    return history, all_preds, all_labels

cnn_history, cnn_preds, cnn_labels = train_model(cnn_model, train_loader_cnn, test_loader_cnn, epochs=8)

## 5. Vision Transformer (ViT) Model

We use a pretrained ViT (`vit_base_patch16_224`) from the `timm` library and fine-tune its classifier head on our dataset.

In [ ]:
train_ds_vit = SignLanguageDataset(train_images, train_labels, vit_transform)
test_ds_vit = SignLanguageDataset(test_images, test_labels, vit_transform)

train_loader_vit = DataLoader(train_ds_vit, batch_size=32, shuffle=True)
test_loader_vit = DataLoader(test_ds_vit, batch_size=32, shuffle=False)

vit_model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=26)
vit_model = vit_model.to(device)

# Freeze the backbone, only fine-tune the classifier head for speed (optional: unfreeze for full fine-tuning)
for name, param in vit_model.named_parameters():
    if 'head' not in name:
        param.requires_grad = False

print('Trainable parameters:', sum(p.numel() for p in vit_model.parameters() if p.requires_grad))

In [ ]:
vit_history, vit_preds, vit_labels = train_model(vit_model, train_loader_vit, test_loader_vit, epochs=5, lr=1e-4)

## 6. Compare CNN vs ViT

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(cnn_history['test_acc'], label='CNN', marker='o')
plt.plot(vit_history['test_acc'], label='ViT', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Test Accuracy')
plt.title('CNN vs ViT — Test Accuracy per Epoch')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final CNN accuracy: {cnn_history['test_acc'][-1]:.4f}")
print(f"Final ViT accuracy: {vit_history['test_acc'][-1]:.4f}")

In [ ]:
# Confusion matrix for the ViT model (usually the stronger one)
cm = confusion_matrix(vit_labels, vit_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap='Blues')
plt.title('ViT Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print(classification_report(vit_labels, vit_preds))

## 7. Save the model

In [ ]:
torch.save(vit_model.state_dict(), 'vit_sign_language_model.pth')
torch.save(cnn_model.state_dict(), 'cnn_sign_language_model.pth')
print('Models saved.')

## Conclusion

- The CNN provides a fast, lightweight baseline trained from scratch.
- The fine-tuned ViT leverages pretrained visual representations and typically achieves higher accuracy with fewer training epochs, since it only needs to adapt its classification head.
- This notebook demonstrates an end-to-end pipeline: data loading, preprocessing, model training, evaluation, and comparison between a CNN and a transformer-based architecture for image classification — directly applicable to sign language gesture recognition.